# 05 — 2D Shallow-Water Simulator and Animator

First SWE2D experiment: build the reference numerical solver workflow, visualize wave propagation, and create the side-by-side animation format that will later compare the solver against a learned CNN dynamics model.

Current reference system is the linearized damped shallow-water model:

```text
eta_t + H (u_x + v_y) = 0
u_t   + g eta_x      = -r u + nu Δu
v_t   + g eta_y      = -r v + nu Δv
```

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / "shallow_water").exists():
            return p
    raise RuntimeError("Start Jupyter from the DeepLearningFTN repository.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from shallow_water import (
    SWE2DConfig,
    cfl_number,
    compute_energy,
    compute_mass,
    compute_rmse,
    gaussian_bump_ic,
    random_bumps_ic,
    simulate,
    velocity_magnitude,
)
from shallow_water.animate import animate_eta_comparison, save_gif

print("repo:", REPO_ROOT)

## Configure solver

In [ ]:
cfg = SWE2DConfig(
    nx=64,
    ny=64,
    lx=1.0,
    ly=1.0,
    dt=0.002,
    gravity=9.81,
    depth=1.0,
    damping=0.10,
    viscosity=1e-4,
    boundary="periodic",
)

STEPS = 300
STORE_EVERY = 1

print(cfg)
print("dx, dy:", cfg.dx, cfg.dy)
print("wave speed:", cfg.wave_speed)
print("CFL estimate:", cfl_number(cfg))

## Simulate one Gaussian wave

In [ ]:
state0 = gaussian_bump_ic(cfg, amplitude=0.10, sigma=0.055, center=(0.50, 0.50))
traj = simulate(state0, cfg, steps=STEPS, store_every=STORE_EVERY)

energy = np.array([compute_energy(s, cfg) for s in traj])
mass = np.array([compute_mass(s, cfg) for s in traj])
t = np.arange(len(traj)) * cfg.dt * STORE_EVERY

print("trajectory:", traj.shape)
print("energy start/end:", energy[0], energy[-1])
print("mass start/end:", mass[0], mass[-1])

## Static frames + diagnostics

In [ ]:
frame_ids = [0, 50, 100, 150, 200, 300]
vmax = np.max(np.abs(traj[:, 0]))

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for ax, idx in zip(axes.ravel(), frame_ids):
    im = ax.imshow(traj[idx, 0], origin="lower", vmin=-vmax, vmax=vmax)
    ax.set_title(f"eta, frame={idx}, t={idx*cfg.dt:.3f}s")
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(t, energy, label="energy")
plt.xlabel("time [s]")
plt.ylabel("energy")
plt.title("Energy should decay because damping is enabled")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(t, mass - mass[0], label="mass drift")
plt.xlabel("time [s]")
plt.ylabel("mass - mass[0]")
plt.title("Mass conservation diagnostic")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Create reference solver animation

In [ ]:
anim = animate_eta_comparison(traj, None, cfg, interval_ms=40, title="Reference linear SWE solver")
out_path = REPO_ROOT / "results" / "swe2d_reference_gaussian.gif"
save_gif(anim, out_path, fps=25)
print("saved:", out_path)

## Side-by-side comparison placeholder

Before training the neural predictor, this gives the final visualization format:

```text
true solver | predicted solver/model | absolute error
```

Here the predicted trajectory is intentionally generated with slightly wrong damping/viscosity. Later this panel will be replaced by CNN or energy-projected CNN.

In [ ]:
pred_cfg = SWE2DConfig(
    nx=cfg.nx,
    ny=cfg.ny,
    lx=cfg.lx,
    ly=cfg.ly,
    dt=cfg.dt,
    gravity=cfg.gravity,
    depth=cfg.depth,
    damping=0.03,       # intentionally wrong
    viscosity=3e-5,     # intentionally wrong
    boundary=cfg.boundary,
)

traj_pred = simulate(state0, pred_cfg, steps=STEPS, store_every=STORE_EVERY)
rmse = compute_rmse(traj, traj_pred)

plt.figure(figsize=(8, 4))
plt.plot(t, rmse)
plt.xlabel("time [s]")
plt.ylabel("RMSE")
plt.title("Reference vs intentionally degraded solver")
plt.grid(True, alpha=0.3)
plt.show()

anim_cmp = animate_eta_comparison(traj, traj_pred, cfg, interval_ms=40, title="Reference vs degraded solver")
out_cmp_path = REPO_ROOT / "results" / "swe2d_reference_vs_degraded.gif"
save_gif(anim_cmp, out_cmp_path, fps=25)
print("saved:", out_cmp_path)

## Next step

After this works, the next notebook should generate a dataset of many trajectories:

```text
state_t = [eta_t, u_t, v_t]
target  = state_{t+1} or xdot_t
```

Then we train:

1. CNN derivative predictor,
2. energy-projected CNN predictor using physical shallow-water energy as Lyapunov function,
3. later: latent autoencoder + ICNN Lyapunov dynamics.